In [1]:
# ========================
# Week 5 - Day 1: OWASP Top 10 & Threats
# ========================

In [2]:
# Fake user database
users_db = {
    "sharuk": "hashed_password_123",
    "admin":  "hashed_password_456"
}

def vulnerable_login(username, password):
    query = f"SELECT * FROM users WHERE username='{username}'"
    print(f"Query executed: {query}")
    
    # Simulate SQL injection
    if "' OR '" in username or "' OR 1" in username:
        print("❌ SQL INJECTION DETECTED IN SIMULATION!")
        return "HACKED — All users exposed!"
    
    if username in users_db:
        return f"Welcome {username}!"
    return "User not found"

# Normal login
print("=== Normal Login ===")
print(vulnerable_login("sharuk", "mypassword"))

# SQL Injection attack
print("\n=== SQL Injection Attack ===")
print(vulnerable_login("admin' OR '1'='1", "anything"))

=== Normal Login ===
Query executed: SELECT * FROM users WHERE username='sharuk'
Welcome sharuk!

=== SQL Injection Attack ===
Query executed: SELECT * FROM users WHERE username='admin' OR '1'='1'
❌ SQL INJECTION DETECTED IN SIMULATION!
HACKED — All users exposed!


In [3]:
import hashlib
import os

# Properly hashed user database
def hash_password(password):
    salt = os.urandom(16).hex()
    hashed = hashlib.sha256((password + salt).encode()).hexdigest()
    return salt, hashed

# Create secure database
secure_db = {}
salt1, hash1 = hash_password("mypassword")
salt2, hash2 = hash_password("adminpass")
secure_db["sharuk"] = {"salt": salt1, "hash": hash1}
secure_db["admin"]  = {"salt": salt2, "hash": hash2}

def secure_login(username, password):
    # Input validation — reject suspicious characters
    forbidden = ["'", '"', ";", "--", "OR", "="]
    for char in forbidden:
        if char in username:
            print(f"❌ Suspicious input detected: {char}")
            return "Access denied!"
    
    # Parameterized query simulation
    # In real SQL: cursor.execute("SELECT * FROM users WHERE username=?", (username,))
    if username not in secure_db:
        return "User not found!"
    
    # Verify password
    user = secure_db[username]
    attempt = hashlib.sha256((password + user["salt"]).encode()).hexdigest()
    
    if attempt == user["hash"]:
        return f"Welcome {username}! ✅"
    return "Wrong password! ❌"

# Test secure login
print("=== Secure Login Tests ===")
print(secure_login("sharuk", "mypassword"))       # Should work
print(secure_login("sharuk", "wrongpassword"))    # Should fail
print(secure_login("admin' OR '1'='1", "x"))     # Should be blocked

=== Secure Login Tests ===
Welcome sharuk! ✅
Wrong password! ❌
❌ Suspicious input detected: '
Access denied!
